In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import wandb
import os

# Initialize wandb
wandb.login()
wandb.init(project="image-colourization-cifar10")

config = wandb.config
config.NIC = 1
config.NC = 24
config.NF = 16
config.kernel_size = 3
config.lr = 1e-3
config.batch_size = 64
config.optimizer_type = "adam"
config.epochs = 25

class CIFAR10Colorization(Dataset):
    def __init__(self, train=True):
        self.dataset = datasets.CIFAR10(root="./data", train=train, download=True)
        self.centroids = np.load("color_centroids.npy").astype(np.float32) / 255.0  # 24x3 array
        self.to_gray = transforms.Grayscale(num_output_channels=1)
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        img_rgb = np.array(img).astype(np.float32) / 255.0  # [32,32,3]

        gray = self.to_gray(img)
        gray = self.to_tensor(gray)  # [1,32,32]

        pixels = img_rgb.reshape(-1, 3)
        dist = np.linalg.norm(pixels[:, None, :] - self.centroids[None, :, :], axis=2)
        label_map = np.argmin(dist, axis=1).reshape(32, 32)
        label_map = torch.from_numpy(label_map).long()

        return gray, label_map


train_dataset = CIFAR10Colorization(train=True)
val_dataset = CIFAR10Colorization(train=False)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

class ColorizationCNN(nn.Module):
    def __init__(self, NIC=1, NF=16, NC=24, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.encoder = nn.Sequential(
            nn.Conv2d(NIC, NF, kernel_size, padding=pad),
            nn.BatchNorm2d(NF),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(NF, 2*NF, kernel_size, padding=pad),
            nn.BatchNorm2d(2*NF),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(2*NF, 4*NF, kernel_size, padding=pad),
            nn.BatchNorm2d(4*NF),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(4*NF, 2*NF, kernel_size=2, stride=2),
            nn.BatchNorm2d(2*NF),
            nn.ReLU(),

            nn.ConvTranspose2d(2*NF, NF, kernel_size=2, stride=2),
            nn.BatchNorm2d(NF),
            nn.ReLU(),

            nn.ConvTranspose2d(NF, NC, kernel_size=2, stride=2),
            nn.BatchNorm2d(NC),
            nn.ReLU(),

            nn.Conv2d(NC, NC, kernel_size=1)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ColorizationCNN(config.NIC, config.NF, config.NC, config.kernel_size).to(device)

criterion = nn.CrossEntropyLoss()
if config.optimizer_type.lower() == "adam":
    optimizer = optim.Adam(model.parameters(), lr=config.lr)
else:
    optimizer = optim.SGD(model.parameters(), lr=config.lr, momentum=0.9)

best_val_loss = float("inf")
save_path = "best_colorization_model.pth"

for epoch in range(config.epochs):
    model.train()
    total_train_loss = 0

    for gray, label_map in train_loader:
        gray, label_map = gray.to(device), label_map.to(device)
        optimizer.zero_grad()
        logits = model(gray)
        loss = criterion(logits, label_map)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for gray, label_map in val_loader:
            gray, label_map = gray.to(device), label_map.to(device)
            logits = model(gray)
            loss = criterion(logits, label_map)
            total_val_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)

    wandb.log({"epoch": epoch+1, "train_loss": avg_train_loss, "val_loss": avg_val_loss})
    print(f"Epoch [{epoch+1}/{config.epochs}] Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print("Saved best model")


model.load_state_dict(torch.load(save_path))
model.eval()
centroids = np.load("color_centroids.npy") / 255.0  # Normalize centroids

def decode_centroid_map(pred_map):
    # pred_map: (32,32) integers in [0,23]
    rgb_img = centroids[pred_map]  # shape: [32,32,3]
    return np.clip(rgb_img, 0, 1)  # Ensure range for imshow

with torch.no_grad():
    for gray, label_map in val_loader:
        gray = gray.to(device)
        logits = model(gray)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        break

fig, axs = plt.subplots(10, 3, figsize=(6, 20))
for i in range(10):
    gray_img = gray[i][0].cpu().numpy()
    pred_color = decode_centroid_map(preds[i])
    gt_color = decode_centroid_map(label_map[i].numpy())

    axs[i, 0].imshow(gray_img, cmap="gray")
    axs[i, 1].imshow(pred_color)
    axs[i, 2].imshow(gt_color)

    axs[i, 0].set_title("Gray")
    axs[i, 1].set_title("Predicted")
    axs[i, 2].set_title("Ground Truth")

    for j in range(3): axs[i, j].axis("off")

plt.tight_layout()
plt.show()

from sympy import symbols

NIC, NF, NC, k = symbols("NIC NF NC k")

weights = (
    NIC*NF*k*k +
    NF*(2*NF)*k*k +
    (2*NF)*(4*NF)*k*k +
    (4*NF)*(2*NF)*(2*2) +
    (2*NF)*(NF)*(2*2) +
    (NF)*(NC)*(2*2) +
    NC*NC*1*1
)
print("Number of weights =", weights.simplify())

outputs = (
    NF*16*16 + 2*NF*8*8 + 4*NF*4*4 +
    2*NF*8*8 + NF*16*16 + NC*32*32 + NC*32*32
)
print("Total activation elements =", outputs.simplify())

connections = weights * (32*32 / NIC)
print("Connections (approx.) =", connections)
print("When input size doubles to 64x64, activations x4.")


sweep_config = {
    "method": "random",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "lr": {"values": [1e-4, 3e-4, 1e-3, 3e-3]},
        "batch_size": {"values": [32, 64, 128]},
        "NF": {"values": [8, 16, 32]},
        "kernel_size": {"values": [3, 5]},
        "optimizer": {"values": ["adam", "sgd"]},
    },
}

sweep_id = wandb.sweep(sweep_config, project="image-colourization-cifar10")
# Then run
wandb.agent(sweep_id)